# 04 Quality Framework

**Input:** `data/processed/03_user_stories_features.csv` from notebook 03.

**Goal:** define the scoring methodology that turns features into quality scores. This notebook does not apply the scoring to data. It defines the rules, formulas, and interpretation bands. Notebook 05 will apply these rules to the 31,394 stories.

**References used in this notebook:**

- Cohn (Mountain Goat Software). The INVEST checklist: Independent, Negotiable, Valuable, Estimatable, Small, Testable.
- Lucassen et al. (2016). The QUS framework defines 13 quality criteria. Each of my dimensions will map to one or more QUS criteria.
- Mordal et al. (2012), Squale model. Provides the aggregation method with hard, medium, and soft weighting (lambda equals 30, 9, 3). I will use the same formula to aggregate per-story scores into per-project and overall scores in notebook 05.
- Challa et al. (2011). Provides the multi-perspective scoring structure (developer, user, project manager) and the interpretation bands (Very Good, Good, Average, Poor, Very Poor). I adapt the perspectives to my dashboard audiences (Product, QA, Project Management).
- Zul et al. (2025) systematic review. Confirms the 8 shared core criteria across frameworks. I make sure each of these is covered by at least one of my dimensions.

**Output:** `src/scoring.py`. A Python module with the scoring functions that notebook 05 will import.

## Quality dimensions

I score each story on five dimensions. Each dimension is a number from 0 to 5, where 5 is the best.

| Dimension | What it measures | Maps to |
|---|---|---|
| Clarity | Is the story unambiguous and follows a recognizable format? | QUS Unambiguous, Full sentence, Well formed |
| Completeness | Does the story have all the parts a reader needs? | QUS Well formed, Minimal, Conceptually sound; Cohn Conditions of Satisfaction |
| Testability | Can QA write a concrete test for this story? | INVEST Testable; QUS Full sentence; Cohn Confirmation |
| Business value | Is the reason for the story stated? | QUS Conceptually sound (means and ends); Cohn template "so that" clause |
| Scope risk | Is the story small enough to fit a sprint? | INVEST Small; QUS Estimatable, Atomic |

The five dimensions are then combined into a single `overall_quality_score`, also on a 0 to 5 scale.

## Issue tag taxonomy

Issue tags are short labels that explain why a score is low. They are inspired by the AQUSA tool (Lucassen et al., 2016) and the failure modes observed by Yamani et al. (2025) on LLM generated stories.

| Tag | Meaning |
|---|---|
| missing_description | Description field is empty after cleaning |
| missing_acceptance_criteria | No "acceptance criteria" phrase, no "given when then", no checklist markers |
| weak_role | No "as a" or "as an" in title or description |
| weak_means | No "I want / need / would like" in title or description |
| missing_reason | No "so that" in title or description |
| has_vague_words | One or more vague adjectives present |
| has_implementation_hint | One or more solution-prescribing terms present |
| non_atomic | More than 3 conjunctions in title plus description |
| missing_estimate | No Story Point assigned |
| non_fibonacci_estimate | Story Point present but off the Planning Poker scale |
| high_scope_risk | Story Point greater than or equal to 13 |
| extreme_scope_risk | Story Point greater than or equal to 40 |
| duplicate_in_project | Exact duplicate inside the same project |
| markup_only | Original description contained only markup, no prose |

## Interpretation bands

Adapted from Challa et al. (2011), rescaled from their 0 to 1 range to my 0 to 5 range:

| Overall score | Band |
|---|---|
| 4.0 to 5.0 | Very good |
| 3.0 to 4.0 | Good |
| 2.0 to 3.0 | Average |
| 1.0 to 2.0 | Poor |
| 0.0 to 1.0 | Very poor |

## Aggregation method (Squale lambda weighting)

When summarizing many stories into a per-project or overall score in notebook 05, I will use the Squale aggregation function from Mordal et al. (2012):

```
ISquale(x1, ..., xn) = -log_lambda( mean( lambda^(-xi) ) )
```

Three weight levels:

| Weight | Lambda | When to use |
|---|---|---|
| Soft | 3 | General overview, gentle to bad components |
| Medium | 9 | Default for most dimensions |
| Hard | 30 | Critical signals (missing acceptance criteria, missing description) |

The function is proven by Mordal et al. to satisfy two important properties:
1. The result is never worse than the smallest input and never better than the arithmetic mean.
2. Any improvement in component quality is reflected in the aggregated score (anti transfers principle).

This notebook stops here. Notebook 05 applies these rules to the 31,394 cleaned stories. 

In [1]:
from typing import Dict, List, Any
def clamp(value: float, low: float = 0.0, high: float = 5.0) -> float:
    return max(low, min(high, value))

In [2]:
def clarity_score(story: Dict[str, Any]) -> float:
    """
    Compute clarity score for one story.
    Higher is better. Range 0 to 5.

    Inputs (relevant columns from the dataframe):
      - is_cohn_full_template  (bool)
      - is_well_formed         (bool)
      - has_as_a               (bool)
      - has_means              (bool)
      - has_so_that            (bool)
      - has_vague_words        (bool)
      - vague_word_count       (int)
      - flag_title_too_short   (bool)
      - flag_title_too_long    (bool)
      - flag_description_markup_only (bool)
    """
    score = 5.0
    if story['is_cohn_full_template']:
        pass  # full template, no penalty
    elif story['is_well_formed']:
        score -= 0.5  # role plus means but no reason
    elif story['has_as_a'] or story['has_means']:
        score -= 1.5  # only partial format
    else:
        score -= 2.5  # no recognizable format at all

    if story['flag_title_too_short']:
        score -= 1.0  # one or two words is rarely informative
    if story['flag_title_too_long']:
        score -= 0.5  # title overload, story stuffed into title
    if story['has_vague_words']:
        # First vague word costs 0.5, additional ones cost 0.25 each
        # Capped so a story with many vague words does not go negative on this alone
        vague_penalty = min(1.5, 0.5 + 0.25 * (story['vague_word_count'] - 1))
        score -= vague_penalty
    if story['flag_description_markup_only']:
        score -= 2.0
    return clamp(score)

In [3]:
story_perfect = {
    'is_cohn_full_template': True,
    'is_well_formed': True,
    'has_as_a': True,
    'has_means': True,
    'has_so_that': True,
    'has_vague_words': False,
    'vague_word_count': 0,
    'flag_title_too_short': False,
    'flag_title_too_long': False,
    'flag_description_markup_only': False,
}
story_bad = {
    'is_cohn_full_template': False,
    'is_well_formed': False,
    'has_as_a': False,
    'has_means': False,
    'has_so_that': False,
    'has_vague_words': True,
    'vague_word_count': 3,
    'flag_title_too_short': True,
    'flag_title_too_long': False,
    'flag_description_markup_only': True,
}

story_middle = {
    'is_cohn_full_template': False,
    'is_well_formed': False,
    'has_as_a': True,
    'has_means': False,
    'has_so_that': False,
    'has_vague_words': True,
    'vague_word_count': 1,
    'flag_title_too_short': False,
    'flag_title_too_long': False,
    'flag_description_markup_only': False,
}
print(f"Perfect story clarity: {clarity_score(story_perfect):.2f} / 5  (expect apr.5.0)")
print(f"Middle story clarity:  {clarity_score(story_middle):.2f} / 5  (expect apro.2.5 to 3.0)")
print(f"Bad story clarity:     {clarity_score(story_bad):.2f} / 5  (expect 0.0 to 1.0)")

Perfect story clarity: 5.00 / 5  (expect apr.5.0)
Middle story clarity:  3.00 / 5  (expect apro.2.5 to 3.0)
Bad story clarity:     0.00 / 5  (expect 0.0 to 1.0)


In [4]:
def completeness_score(story: Dict[str, Any]) -> float:
    """
    Compute completeness score for one story.
    Higher is better. Range 0 to 5.

    Inputs:
      - flag_description_missing      (bool)
      - flag_description_too_short    (bool)
      - has_acceptance_criteria       (bool)
      - has_as_a, has_means, has_so_that (bool)
      - description_word_count        (int)
    """
    score = 5.0
    if story['flag_description_missing']:
        score -= 2.5  # severe, no description at all
    elif story['flag_description_too_short']:
        score -= 1.0  # description present but under 10 words
    if not story['has_acceptance_criteria']:
        score -= 1.5  # no AC explicit, hard to mark a story "done"
    if not story['has_as_a']:
        score -= 0.5  # who is the user
    if not story['has_means']:
        score -= 0.5  # what do they want
    if not story['has_so_that']:
        score -= 0.5  # why do they want it
    return clamp(score)

In [5]:
# testability_score
# Question: can QA write a concrete test for this story?
# Maps to:
#   - INVEST "Testable" (Cohn)
#   - QUS "Full sentence" (clear enough to derive tests)
#   - Cohn "Confirmation" pillar (acceptance criteria are tests of done)
#
# A story is testable when:
#   - QA knows what "done" looks like (acceptance criteria)
#   - The wording is not vague (no "fast", "easy", "better")
#   - The story is not a dump of code or markup

def testability_score(story: Dict[str, Any]) -> float:
    """
    Compute testability score for one story.
    Higher is better. Range 0 to 5.

    Inputs:
      - has_acceptance_criteria       (bool)
      - has_vague_words               (bool)
      - vague_word_count              (int)
      - flag_description_missing      (bool)
      - flag_description_markup_only  (bool)
      - has_means                     (bool)
    """
    score = 5.0
    if not story['has_acceptance_criteria']:
        score -= 2.0
    if story['has_vague_words']:
        vague_penalty = min(2.0, 0.75 + 0.5 * (story['vague_word_count'] - 1))
        score -= vague_penalty
    if story['flag_description_missing']:
        score -= 1.5
    if story['flag_description_markup_only']:
        score -= 2.0
    if not story['has_means']:
        score -= 0.5

    return clamp(score)

In [6]:
# business_value_score
# Question: is the reason for this story explicit?
# Maps to:
#   - Cohn template "so that <reason>" clause
#   - QUS "Conceptually sound" (means and ends fit together)
#   - INVEST "Valuable"
def business_value_score(story: Dict[str, Any]) -> float:
    """
    Compute business value score for one story.
    Higher is better. Range 0 to 5.

    Inputs:
      - has_so_that                   (bool)
      - has_as_a                      (bool)
      - has_means                     (bool)
      - flag_description_missing      (bool)
      - has_implementation_hint       (bool)
    """
    score = 5.0
    if not story['has_so_that']:
        score -= 2.5  # heaviest penalty in this dimension
    if not story['has_as_a']:
        score -= 1.0
    if not story['has_means']:
        score -= 0.5
    if story['flag_description_missing']:
        score -= 1.5
    if story['has_implementation_hint']:
        score -= 0.5

    return clamp(score)

In [7]:
def scope_risk_score(story: Dict[str, Any]) -> float:
    """
    Compute scope risk score for one story.
    Higher is better (= lower risk). Range 0 to 5.

    Inputs:
      - flag_sp_missing               (bool)
      - flag_sp_high_scope_risk       (bool, SP >= 13)
      - flag_sp_extreme_scope_risk    (bool, SP >= 40)
      - is_fibonacci_sp               (bool)
      - has_multi_feature_signal      (bool, conjunctions > 3)
      - conjunction_count             (int)
    """
    score = 5.0
    if story['flag_sp_missing']:
        score -= 1.5  # cannot judge size without an estimate
    else:
        if story['flag_sp_extreme_scope_risk']:
            score -= 3.0  # an epic mislabeled as a story
        elif story['flag_sp_high_scope_risk']:
            score -= 1.5  # too big for one sprint
        if not story['is_fibonacci_sp']:
            score -= 0.5
    if story['has_multi_feature_signal']:
        atomic_penalty = min(2.0, 0.5 + 0.25 * (story['conjunction_count'] - 3))
        score -= atomic_penalty

    return clamp(score)

In [9]:
story_perfect_full = {
    **story_perfect,
    'flag_description_missing': False,
    'flag_description_too_short': False,
    'has_acceptance_criteria': True,
    'description_word_count': 50,
    'has_implementation_hint': False,
    'flag_sp_missing': False,
    'flag_sp_high_scope_risk': False,
    'flag_sp_extreme_scope_risk': False,
    'is_fibonacci_sp': True,
    'has_multi_feature_signal': False,
    'conjunction_count': 1,
}

story_bad_full = {
    **story_bad,
    'flag_description_missing': True,
    'flag_description_too_short': False,
    'has_acceptance_criteria': False,
    'description_word_count': 0,
    'has_implementation_hint': True,
    'flag_sp_missing': True,
    'flag_sp_high_scope_risk': False,
    'flag_sp_extreme_scope_risk': False,
    'is_fibonacci_sp': False,
    'has_multi_feature_signal': True,
    'conjunction_count': 8,
}

story_middle_full = {
    **story_middle,
    'flag_description_missing': False,
    'flag_description_too_short': True,
    'has_acceptance_criteria': False,
    'description_word_count': 25,
    'has_implementation_hint': False,
    'flag_sp_missing': False,
    'flag_sp_high_scope_risk': False,
    'flag_sp_extreme_scope_risk': False,
    'is_fibonacci_sp': True,
    'has_multi_feature_signal': False,
    'conjunction_count': 2,
}

def print_scores(label, story):
    print(f"\n{label}")
    print("-" * 40)
    print(f"  clarity:        {clarity_score(story):.2f} / 5")
    print(f"  completeness:   {completeness_score(story):.2f} / 5")
    print(f"  testability:    {testability_score(story):.2f} / 5")
    print(f"  business_value: {business_value_score(story):.2f} / 5")
    print(f"  scope_risk:     {scope_risk_score(story):.2f} / 5  (higher = lower risk)")
print_scores("Perfect story", story_perfect_full)
print_scores("Middle story",  story_middle_full)
print_scores("Bad story",     story_bad_full)


Perfect story
----------------------------------------
  clarity:        5.00 / 5
  completeness:   5.00 / 5
  testability:    5.00 / 5
  business_value: 5.00 / 5
  scope_risk:     5.00 / 5  (higher = lower risk)

Middle story
----------------------------------------
  clarity:        3.00 / 5
  completeness:   1.50 / 5
  testability:    1.75 / 5
  business_value: 2.00 / 5
  scope_risk:     5.00 / 5  (higher = lower risk)

Bad story
----------------------------------------
  clarity:        0.00 / 5
  completeness:   0.00 / 5
  testability:    0.00 / 5
  business_value: 0.00 / 5
  scope_risk:     1.75 / 5  (higher = lower risk)


In [10]:
# overall_quality_score
# Question: what is the single quality score for this story?
# Approach:
#   Take the five dimension scores and combine them. The Squale paper
#   (Mordal et al., 2012) showed that plain averaging hides bad
#   components. So we use a generalised mean that puts more weight on
#   the lower dimension scores. A story with one very low dimension
#   should get a lower overall score than a story whose dimensions are
#   evenly mediocre.
#
# Formula:
#   For p < 1, the generalised mean of values x1, ..., xn is
#       Mp(x1, ..., xn) = ( mean(xi^p) )^(1/p)
#   The smaller p is, the more the mean is pulled toward the smallest xi.
#   - p = 1: arithmetic mean (no bias).
#   - p = 0: geometric mean (gentle bias toward low values).
#   - p < 0: harmonic-like (strong bias toward low values).
#
# I use p = 0 (geometric mean) by default. It is a known good middle
# ground between arithmetic mean and Squale's full lambda-weighted
# aggregation, and it is easier to explain to non-statistical readers.
#
# Note on direction:
#   All five dimensions are "higher is better". scope_risk is already
#   inverted (5 = low risk). So we can combine them directly.
import math
def overall_quality_score(story: Dict[str, Any]) -> float:
    dimensions = [
        clarity_score(story),
        completeness_score(story),
        testability_score(story),
        business_value_score(story),
        scope_risk_score(story),
    ]
    eps = 0.01
    safe_dims = [max(d, eps) for d in dimensions]
    log_mean = sum(math.log(d) for d in safe_dims) / len(safe_dims)
    geo_mean = math.exp(log_mean)

    return clamp(geo_mean)

In [11]:
print(f"Perfect story overall: {overall_quality_score(story_perfect_full):.2f} / 5  (expect ~5.0)")
print(f"Middle story overall:  {overall_quality_score(story_middle_full):.2f} / 5  (expect ~2.5 to 3.0)")
print(f"Bad story overall:     {overall_quality_score(story_bad_full):.2f} / 5  (expect ~0.0 to 0.5)")
def arithmetic_overall(story):
    dims = [
        clarity_score(story),
        completeness_score(story),
        testability_score(story),
        business_value_score(story),
        scope_risk_score(story),
    ]
    return sum(dims) / len(dims)

print()
print("Comparison: geometric mean vs arithmetic mean")
print("-" * 50)
for label, story in [("Perfect", story_perfect_full),
                     ("Middle",  story_middle_full),
                     ("Bad",     story_bad_full)]:
    geo = overall_quality_score(story)
    arith = arithmetic_overall(story)
    print(f"{label:<8} geometric={geo:.2f}  arithmetic={arith:.2f}  difference={arith - geo:.2f}")

Perfect story overall: 5.00 / 5  (expect ~5.0)
Middle story overall:  2.39 / 5  (expect ~2.5 to 3.0)
Bad story overall:     0.03 / 5  (expect ~0.0 to 0.5)

Comparison: geometric mean vs arithmetic mean
--------------------------------------------------
Perfect  geometric=5.00  arithmetic=5.00  difference=0.00
Middle   geometric=2.39  arithmetic=2.65  difference=0.26
Bad      geometric=0.03  arithmetic=0.35  difference=0.32


In [12]:
# issue_tags
# Question: which specific problems make this story low quality?
# Output: a list of short labels (strings). Each label points to a
# specific failure mode. The dashboard can group stories by tag,
# count the most common tags, or filter by tag.
# Grounded in:
#   - Lucassen et al. (2016), AQUSA tool issue categorisation


def issue_tags(story: Dict[str, Any]) -> List[str]:
    tags = []

    if story.get('flag_description_missing'):
        tags.append('missing_description')
    if story.get('flag_description_markup_only'):
        tags.append('markup_only')
    if not story.get('has_acceptance_criteria'):
        tags.append('missing_acceptance_criteria')
    if not story.get('has_as_a'):
        tags.append('weak_role')
    if not story.get('has_means'):
        tags.append('weak_means')
    if not story.get('has_so_that'):
        tags.append('missing_reason')
    if story.get('has_vague_words'):
        tags.append('has_vague_words')
    if story.get('has_implementation_hint'):
        tags.append('has_implementation_hint')

    if story.get('has_multi_feature_signal'):
        tags.append('non_atomic')

    if story.get('flag_sp_missing'):
        tags.append('missing_estimate')
    else:
        if not story.get('is_fibonacci_sp'):
            tags.append('non_fibonacci_estimate')
        if story.get('flag_sp_extreme_scope_risk'):
            tags.append('extreme_scope_risk')
        elif story.get('flag_sp_high_scope_risk'):
            tags.append('high_scope_risk')
    if story.get('flag_duplicate_in_project'):
        tags.append('duplicate_in_project')

    return tags

In [13]:
print("Perfect story tags:")
print(f"  {issue_tags(story_perfect_full)}")
print()
print("Middle story tags:")
print(f"  {issue_tags(story_middle_full)}")
print()
print("Bad story tags:")
print(f"  {issue_tags(story_bad_full)}")
story_bad_full['flag_duplicate_in_project'] = True
print()
print("Bad story with duplicate flag tags:")
print(f"  {issue_tags(story_bad_full)}")

Perfect story tags:
  []

Middle story tags:
  ['missing_acceptance_criteria', 'weak_means', 'missing_reason', 'has_vague_words']

Bad story tags:
  ['missing_description', 'markup_only', 'missing_acceptance_criteria', 'weak_role', 'weak_means', 'missing_reason', 'has_vague_words', 'has_implementation_hint', 'non_atomic', 'missing_estimate']

Bad story with duplicate flag tags:
  ['missing_description', 'markup_only', 'missing_acceptance_criteria', 'weak_role', 'weak_means', 'missing_reason', 'has_vague_words', 'has_implementation_hint', 'non_atomic', 'missing_estimate', 'duplicate_in_project']


In [14]:
def score_band(score: float) -> str:
    if score >= 4.0:
        return 'Very good'
    if score >= 3.0:
        return 'Good'
    if score >= 2.0:
        return 'Average'
    if score >= 1.0:
        return 'Poor'
    return 'Very poor'
#for the test
for label, story in [("Perfect", story_perfect_full),
                     ("Middle",  story_middle_full),
                     ("Bad",     story_bad_full)]:
    s = overall_quality_score(story)
    band = score_band(s)
    print(f"{label:<8} score={s:.2f}  band={band}")

Perfect  score=5.00  band=Very good
Middle   score=2.39  band=Average
Bad      score=0.03  band=Very poor


In [15]:
# Export all scoring functions to src/scoring.py
SCORING_MODULE = '''"""
Scoring functions for user story quality.

This module is generated from notebook 04. Do not edit by hand. If you
need to change the scoring logic, update notebook 04 and re-export.

The scoring framework follows:
  - Cohn (Mountain Goat Software) - the classic As a / I want / so that
    template and the INVEST checklist.
  - Lucassen et al. (2016) - the QUS framework, 13 quality criteria.
  - Mordal et al. (2012), Squale model - aggregation principles.
  - Challa et al. (2011) - interpretation bands.

Each score is in the range 0 to 5, higher is better. scope_risk is
inverted internally so that 5 means low risk, in line with the other
dimensions.
"""

from typing import Dict, List, Any
import math


def clamp(value: float, low: float = 0.0, high: float = 5.0) -> float:
    """Bound a score to the [low, high] range."""
    return max(low, min(high, value))


def clarity_score(story: Dict[str, Any]) -> float:
    """Clarity: how unambiguous and well-formed is the story? 0 to 5."""
    score = 5.0
    if story['is_cohn_full_template']:
        pass
    elif story['is_well_formed']:
        score -= 0.5
    elif story['has_as_a'] or story['has_means']:
        score -= 1.5
    else:
        score -= 2.5
    if story['flag_title_too_short']:
        score -= 1.0
    if story['flag_title_too_long']:
        score -= 0.5
    if story['has_vague_words']:
        vague_penalty = min(1.5, 0.5 + 0.25 * (story['vague_word_count'] - 1))
        score -= vague_penalty
    if story['flag_description_markup_only']:
        score -= 2.0
    return clamp(score)


def completeness_score(story: Dict[str, Any]) -> float:
    """Completeness: are all the parts present? 0 to 5."""
    score = 5.0
    if story['flag_description_missing']:
        score -= 2.5
    elif story['flag_description_too_short']:
        score -= 1.0
    if not story['has_acceptance_criteria']:
        score -= 1.5
    if not story['has_as_a']:
        score -= 0.5
    if not story['has_means']:
        score -= 0.5
    if not story['has_so_that']:
        score -= 0.5
    return clamp(score)


def testability_score(story: Dict[str, Any]) -> float:
    """Testability: can QA write a concrete test? 0 to 5."""
    score = 5.0
    if not story['has_acceptance_criteria']:
        score -= 2.0
    if story['has_vague_words']:
        vague_penalty = min(2.0, 0.75 + 0.5 * (story['vague_word_count'] - 1))
        score -= vague_penalty
    if story['flag_description_missing']:
        score -= 1.5
    if story['flag_description_markup_only']:
        score -= 2.0
    if not story['has_means']:
        score -= 0.5
    return clamp(score)


def business_value_score(story: Dict[str, Any]) -> float:
    """Business value: is the reason explicit? 0 to 5."""
    score = 5.0
    if not story['has_so_that']:
        score -= 2.5
    if not story['has_as_a']:
        score -= 1.0
    if not story['has_means']:
        score -= 0.5
    if story['flag_description_missing']:
        score -= 1.5
    if story['has_implementation_hint']:
        score -= 0.5
    return clamp(score)


def scope_risk_score(story: Dict[str, Any]) -> float:
    """Scope risk: is the story small enough? Higher = lower risk. 0 to 5."""
    score = 5.0
    if story['flag_sp_missing']:
        score -= 1.5
    else:
        if story['flag_sp_extreme_scope_risk']:
            score -= 3.0
        elif story['flag_sp_high_scope_risk']:
            score -= 1.5
        if not story['is_fibonacci_sp']:
            score -= 0.5
    if story['has_multi_feature_signal']:
        atomic_penalty = min(2.0, 0.5 + 0.25 * (story['conjunction_count'] - 3))
        score -= atomic_penalty
    return clamp(score)


def overall_quality_score(story: Dict[str, Any]) -> float:
    """Combine the five dimensions into one score using geometric mean."""
    dimensions = [
        clarity_score(story),
        completeness_score(story),
        testability_score(story),
        business_value_score(story),
        scope_risk_score(story),
    ]
    eps = 0.01
    safe_dims = [max(d, eps) for d in dimensions]
    log_mean = sum(math.log(d) for d in safe_dims) / len(safe_dims)
    geo_mean = math.exp(log_mean)
    return clamp(geo_mean)


def issue_tags(story: Dict[str, Any]) -> List[str]:
    """Return a list of issue tag strings that apply to this story."""
    tags = []
    if story.get('flag_description_missing'):
        tags.append('missing_description')
    if story.get('flag_description_markup_only'):
        tags.append('markup_only')
    if not story.get('has_acceptance_criteria'):
        tags.append('missing_acceptance_criteria')
    if not story.get('has_as_a'):
        tags.append('weak_role')
    if not story.get('has_means'):
        tags.append('weak_means')
    if not story.get('has_so_that'):
        tags.append('missing_reason')
    if story.get('has_vague_words'):
        tags.append('has_vague_words')
    if story.get('has_implementation_hint'):
        tags.append('has_implementation_hint')
    if story.get('has_multi_feature_signal'):
        tags.append('non_atomic')
    if story.get('flag_sp_missing'):
        tags.append('missing_estimate')
    else:
        if not story.get('is_fibonacci_sp'):
            tags.append('non_fibonacci_estimate')
        if story.get('flag_sp_extreme_scope_risk'):
            tags.append('extreme_scope_risk')
        elif story.get('flag_sp_high_scope_risk'):
            tags.append('high_scope_risk')
    if story.get('flag_duplicate_in_project'):
        tags.append('duplicate_in_project')
    return tags


def score_band(score: float) -> str:
    """Convert a numeric quality score (0 to 5) to a readable band."""
    if score >= 4.0:
        return 'Very good'
    if score >= 3.0:
        return 'Good'
    if score >= 2.0:
        return 'Average'
    if score >= 1.0:
        return 'Poor'
    return 'Very poor'
'''

from pathlib import Path
out = Path('../src/scoring.py')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(SCORING_MODULE, encoding='utf-8')
print(f"Exported scoring module to: {out}")
print(f"Size: {out.stat().st_size:,} bytes")
import importlib.util
spec = importlib.util.spec_from_file_location("scoring", out)
scoring_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scoring_module)
print()
print("Verification: re-import and run on perfect story")
print(f"  clarity:   {scoring_module.clarity_score(story_perfect_full):.2f}")
print(f"  overall:   {scoring_module.overall_quality_score(story_perfect_full):.2f}")
print(f"  band:      {scoring_module.score_band(scoring_module.overall_quality_score(story_perfect_full))}")

Exported scoring module to: ..\src\scoring.py
Size: 6,050 bytes

Verification: re-import and run on perfect story
  clarity:   5.00
  overall:   5.00
  band:      Very good


## Summary

**What this notebook did:**
- Defined the scoring methodology for user story quality.
- Built 5 dimension scoring functions: clarity, completeness, testability, business_value, scope_risk.
- Each dimension returns a score in the range 0 to 5, higher is better.
- Combined the 5 dimensions into one `overall_quality_score` using a geometric mean. The geometric mean penalises stories with a single very weak dimension more than the arithmetic mean would, in line with Mordal et al. (2012).
- Added an `issue_tags` function that produces a list of specific failure labels per story.
- Added a `score_band` function that maps a numeric score to a readable category (Very good, Good, Average, Poor, Very poor) following Challa et al. (2011).
- Exported all of the above to `src/scoring.py` as a reusable Python module.

**Why the geometric mean:**
Mordal et al. (2012) show that the arithmetic mean hides bad components. We verified this on three test stories. For the bad test story the arithmetic mean gives 0.35 (looks like "Very poor" but not extreme), while the geometric mean gives 0.03 (clearly extreme). The geometric mean better reflects the impact of a single dimension failing badly.

**Failure penalties per dimension:**

| Dimension | Heaviest penalty |
|---|---|
| Clarity | No template at all: minus 2.5. Markup only: minus 2.0. |
| Completeness | No description: minus 2.5. No acceptance criteria: minus 1.5. |
| Testability | No acceptance criteria: minus 2.0. Vague words: up to minus 2.0. |
| Business value | No "so that": minus 2.5. No "as a": minus 1.0. |
| Scope risk | Extreme SP (>= 40): minus 3.0. Multi-feature: up to minus 2.0. |

**Test verification:**
The three test stories (Perfect, Middle, Bad) produced scores 5.00, 2.39, 0.03 and bands Very good, Average, Very poor.

**What is next:** notebook 05 imports `src/scoring.py` and applies these functions to all 31,394 stories, producing a scored CSV with overall_quality_score and issue_tags columns. Then I will be able to look at the score distribution, the most common issue tags, and per-project differences.